[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shpark-inco/sch-workshop/blob/main/02_alphafold2_structure_prediction_v0.2.ipynb)

<img src="https://raw.githubusercontent.com/sokrypton/ColabFold/main/.github/ColabFold_Marv_Logo_Small.png" height="200" align="right" style="height:240px">

## ColabFold v1.6.1: MMseqs2를 이용한 AlphaFold2

[AlphaFold2](https://www.nature.com/articles/s41586-021-03819-2)와 [Alphafold2-multimer](https://www.biorxiv.org/content/10.1101/2021.10.04.463034v1)를 이용한, 사용하기 쉬운 단백질 구조·복합체 예측 도구입니다. 서열 정렬/템플릿은 [MMseqs2](https://mmseqs.com)와 [HHsearch](https://github.com/soedinglab/hh-suite)로 생성됩니다. 자세한 내용은 노트북 <a href="#Instructions">하단의 사용법</a>, [ColabFold GitHub](https://github.com/sokrypton/ColabFold), [Nature Protocols](https://www.nature.com/articles/s41596-024-01060-5)를 참고하세요.

이전 버전: [v1.4](https://colab.research.google.com/github/sokrypton/ColabFold/blob/v1.4.0/AlphaFold2.ipynb), [v1.5.1](https://colab.research.google.com/github/sokrypton/ColabFold/blob/v1.5.1/AlphaFold2.ipynb), [v1.5.2](https://colab.research.google.com/github/sokrypton/ColabFold/blob/v1.5.2/AlphaFold2.ipynb), [v1.5.3-patch](https://colab.research.google.com/github/sokrypton/ColabFold/blob/56c72044c7d51a311ca99b953a71e552fdc042e1/AlphaFold2.ipynb)

[Mirdita M, Schütze K, Moriwaki Y, Heo L, Ovchinnikov S, Steinegger M. ColabFold: Making protein folding accessible to all.
*Nature Methods*, 2022](https://www.nature.com/articles/s41592-022-01488-1)

> 🇰🇷 한국어 번역본 v0.1 (sch-workshop) — 코드는 원본 그대로이며 설명만 번역했습니다.

In [ ]:
#@title 단백질 서열을 입력한 뒤 `런타임` -> `모두 실행`을 누르세요
from google.colab import files
import os
import re
import hashlib
import random

from sys import version_info
python_version = f"{version_info.major}.{version_info.minor}"

def add_hash(x,y):
  return x+"_"+hashlib.sha1(y.encode()).hexdigest()[:5]

query_sequence = 'PIAQIHILEGRSDEQKETLIREVSEAISRSLDAPLTSVRVIITEMAKGHFGIGGELASK' #@param {type:"string"}
#@markdown  - **복합체 모델링** 시 단백질 간 사슬 분리는 `:` 로 지정합니다 (동종/이종 올리고머 지원). 예: 동종이량체는 **PI...SK:PI...SK**
jobname = 'test' #@param {type:"string"}
# 사용할 모델 수
num_relax = 0 #@param [0, 1, 5] {type:"raw"}
#@markdown - 상위 랭크 구조 중 몇 개를 amber로 relax(이완)할지 지정
template_mode = "none" #@param ["none", "pdb100","custom"]
#@markdown - `none` = 템플릿 미사용. `pdb100` = pdb100에서 템플릿 탐지([설명](#pdb100)). `custom` = 직접 템플릿 업로드(PDB/mmCIF, [설명](#custom_templates))

use_amber = num_relax > 0

# 공백 제거
query_sequence = "".join(query_sequence.split())

basejobname = "".join(jobname.split())
basejobname = re.sub(r'\W+', '', basejobname)
jobname = add_hash(basejobname, query_sequence)

# 작업 이름 디렉토리가 이미 있는지 확인
def check(folder):
  if os.path.exists(folder):
    return False
  else:
    return True
if not check(jobname):
  n = 0
  while not check(f"{jobname}_{n}"): n += 1
  jobname = f"{jobname}_{n}"

# 결과를 저장할 디렉토리 생성
os.makedirs(jobname, exist_ok=True)

# 쿼리(서열) 저장
queries_path = os.path.join(jobname, f"{jobname}.csv")
with open(queries_path, "w") as text_file:
  text_file.write(f"id,sequence\n{jobname},{query_sequence}")

if template_mode == "pdb100":
  use_templates = True
  custom_template_path = None
elif template_mode == "custom":
  custom_template_path = os.path.join(jobname,f"template")
  os.makedirs(custom_template_path, exist_ok=True)
  uploaded = files.upload()
  use_templates = True
  for fn in uploaded.keys():
    os.rename(fn,os.path.join(custom_template_path,fn))
else:
  custom_template_path = None
  use_templates = False

print("jobname",jobname)
print("sequence",query_sequence)
print("length",len(query_sequence.replace(":","")))

In [ ]:
#@title 의존성 설치
%%time
import os
USE_AMBER = use_amber
USE_TEMPLATES = use_templates
PYTHON_VERSION = python_version

if not os.path.isfile("COLABFOLD_READY"):
  print("colabfold 설치 중...")
  os.system("pip install -q --no-warn-conflicts 'colabfold[alphafold-minus-jax] @ git+https://github.com/sokrypton/ColabFold'")
  if os.environ.get('TPU_NAME', False) != False:
    os.system("pip uninstall -y jax jaxlib")
    os.system("pip install --no-warn-conflicts --upgrade dm-haiku==0.0.10 'jax[cuda12_pip]'==0.3.25 -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html")
  os.system("ln -s /usr/local/lib/python3.*/dist-packages/colabfold colabfold")
  os.system("ln -s /usr/local/lib/python3.*/dist-packages/alphafold alphafold")
  # TensorFlow 충돌을 막기 위한 처리 (TF lite .so 제거)
  os.system("rm -f /usr/local/lib/python3.*/dist-packages/tensorflow/core/kernels/libtfkernel_sobol_op.so /usr/local/lib/python3.*/dist-packages/tensorflow/lite/python/*/*.so")
  os.system("touch COLABFOLD_READY")

if USE_AMBER or USE_TEMPLATES:
  if not os.path.isfile("CONDA_READY"):
    print("conda 설치 중...")
    os.system("wget -qnc https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh")
    os.system("bash Miniforge3-Linux-x86_64.sh -bfp /usr/local")
    os.system("mamba config --set auto_update_conda false")
    os.system("touch CONDA_READY")

if USE_TEMPLATES and not os.path.isfile("HH_READY") and USE_AMBER and not os.path.isfile("AMBER_READY"):
  print("hhsuite와 amber 설치 중...")
  os.system(f"mamba install -y -c conda-forge -c bioconda kalign2=2.04 hhsuite=3.3.0 openmm=8.2.0 python='{PYTHON_VERSION}' pdbfixer")
  os.system("touch HH_READY")
  os.system("touch AMBER_READY")
else:
  if USE_TEMPLATES and not os.path.isfile("HH_READY"):
    print("hhsuite 설치 중...")
    os.system(f"mamba install -y -c conda-forge -c bioconda kalign2=2.04 hhsuite=3.3.0 python='{PYTHON_VERSION}'")
    os.system("touch HH_READY")
  if USE_AMBER and not os.path.isfile("AMBER_READY"):
    print("amber 설치 중...")
    os.system(f"mamba install -y -c conda-forge openmm=8.2.0 python='{PYTHON_VERSION}' pdbfixer")
    os.system("touch AMBER_READY")

In [ ]:
#@markdown ### MSA 옵션 (맞춤 MSA 업로드, 단일 서열, 짝짓기 모드)
msa_mode = "mmseqs2_uniref_env" #@param ["mmseqs2_uniref_env", "mmseqs2_uniref","single_sequence","custom"]
pair_mode = "unpaired_paired" #@param ["unpaired_paired","paired","unpaired"] {type:"string"}
#@markdown - "unpaired_paired" = 같은 종의 서열 짝짓기 + 짝짓지 않은 MSA, "unpaired" = 사슬마다 별도 MSA, "paired" = 짝지어진 서열만 사용.

# 사용할 a3m 결정
if "mmseqs2" in msa_mode:
  a3m_file = os.path.join(jobname,f"{jobname}.a3m")

elif msa_mode == "custom":
  a3m_file = os.path.join(jobname,f"{jobname}.custom.a3m")
  if not os.path.isfile(a3m_file):
    custom_msa_dict = files.upload()
    custom_msa = list(custom_msa_dict.keys())[0]
    header = 0
    import fileinput
    for line in fileinput.FileInput(custom_msa,inplace=1):
      if line.startswith(">"):
         header = header + 1
      if not line.rstrip():
        continue
      if line.startswith(">") == False and header == 1:
         query_sequence = line.rstrip()
      print(line, end='')

    os.rename(custom_msa, a3m_file)
    queries_path=a3m_file
    print(f"moving {custom_msa} to {a3m_file}")

else:
  a3m_file = os.path.join(jobname,f"{jobname}.single_sequence.a3m")
  with open(a3m_file, "w") as text_file:
    text_file.write(">1\n%s" % query_sequence)

In [ ]:
#@markdown ### 고급 설정
model_type = "auto" #@param ["auto", "alphafold2_ptm", "alphafold2_multimer_v1", "alphafold2_multimer_v2", "alphafold2_multimer_v3", "deepfold_v1", "alphafold2"]
#@markdown - `auto` 선택 시 단량체(monomer) 예측에는 `alphafold2_ptm`, 복합체(complex) 예측에는 `alphafold2_multimer_v3` 사용.
#@markdown 어떤 model_type이든 (입력이 단량체든 복합체든) 사용 가능.
num_recycles = "3" #@param ["auto", "0", "1", "3", "6", "12", "24", "48"]
#@markdown - `auto` 선택 시 `model_type=alphafold2_multimer_v3`이면 `num_recycles=20`, 그 외에는 `num_recycles=3`.
recycle_early_stop_tolerance = "auto" #@param ["auto", "0.0", "0.5", "1.0"]
#@markdown - `auto` 선택 시 `model_type=alphafold2_multimer_v3`이면 `tol=0.5`, 그 외에는 `tol=0.0`.
relax_max_iterations = 200 #@param [0, 200, 2000] {type:"raw"}
#@markdown - 최대 amber relax 반복 횟수, `0` = 무제한 (AlphaFold2 기본값, 매우 오래 걸릴 수 있음)
pairing_strategy = "greedy" #@param ["greedy", "complete"] {type:"string"}
#@markdown - `greedy` = 분류학적으로 일치하는 부분집합 짝짓기, `complete` = 모든 서열이 한 줄에서 일치해야 함.
calc_extra_ptm = False #@param {type:"boolean"}
#@markdown - 사슬 쌍별 iptm/actifptm 반환

#@markdown #### 샘플 설정
#@markdown -  모델의 불확실성으로부터 예측을 샘플링하려면 드롭아웃을 활성화하고 시드 수를 늘리세요.
#@markdown -  불확실성을 높이려면 `max_msa`를 줄이세요.
max_msa = "auto" #@param ["auto", "512:1024", "256:512", "64:128", "32:64", "16:32"]
num_seeds = 1 #@param [1,2,4,8,16] {type:"raw"}
use_dropout = False #@param {type:"boolean"}

num_recycles = None if num_recycles == "auto" else int(num_recycles)
recycle_early_stop_tolerance = None if recycle_early_stop_tolerance == "auto" else float(recycle_early_stop_tolerance)
if max_msa == "auto": max_msa = None

#@markdown #### 저장 설정
save_all = False #@param {type:"boolean"}
save_recycles = False #@param {type:"boolean"}
save_to_google_drive = False #@param {type:"boolean"}
#@markdown -  save_to_google_drive 옵션을 선택하면 결과 zip이 Google Drive에 업로드됩니다.
dpi = 200 #@param {type:"integer"}
#@markdown - 이미지 해상도(dpi) 설정

if save_to_google_drive:
  from pydrive2.drive import GoogleDrive
  from pydrive2.auth import GoogleAuth
  from google.colab import auth
  from oauth2client.client import GoogleCredentials
  auth.authenticate_user()
  gauth = GoogleAuth()
  gauth.credentials = GoogleCredentials.get_application_default()
  drive = GoogleDrive(gauth)
  print("Google Drive에 로그인되었습니다. 진행 준비 완료!")

#@markdown 양식을 업데이트한 후 `런타임` -> `모두 실행`을 누르는 것을 잊지 마세요.

In [ ]:
#@title 예측 실행
display_images = True #@param {type:"boolean"}

import sys
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
from Bio import BiopythonDeprecationWarning
warnings.simplefilter(action='ignore', category=BiopythonDeprecationWarning)
from pathlib import Path
from colabfold.download import download_alphafold_params, default_data_dir
from colabfold.utils import setup_logging
from colabfold.batch import get_queries, run, set_model_type
from colabfold.plot import plot_msa_v2

import os
import numpy as np
try:
  K80_chk = os.popen('nvidia-smi | grep "Tesla K80" | wc -l').read()
except:
  K80_chk = "0"
  pass
if "1" in K80_chk:
  print("경고: GPU Tesla K80 발견: 총 길이 < 1000으로 제한됨")
  if "TF_FORCE_UNIFIED_MEMORY" in os.environ:
    del os.environ["TF_FORCE_UNIFIED_MEMORY"]
  if "XLA_PYTHON_CLIENT_MEM_FRACTION" in os.environ:
    del os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"]

from colabfold.colabfold import plot_protein
from pathlib import Path
import matplotlib.pyplot as plt

# pdbfixer import를 위해 필요한 처리
if use_amber and f"/usr/local/lib/python{python_version}/site-packages/" not in sys.path:
    sys.path.insert(0, f"/usr/local/lib/python{python_version}/site-packages/")

# 입력 특징(MSA)을 시각화하는 콜백
def input_features_callback(input_features):
  if display_images:
    plot_msa_v2(input_features)
    plt.show()
    plt.close()

# 예측된 단백질 구조를 시각화하는 콜백
def prediction_callback(protein_obj, length,
                        prediction_result, input_features, mode):
  model_name, relaxed = mode
  if not relaxed:
    if display_images:
      fig = plot_protein(protein_obj, Ls=length, dpi=150)
      plt.show()
      plt.close()

result_dir = jobname
log_filename = os.path.join(jobname,"log.txt")
setup_logging(Path(log_filename))

queries, is_complex = get_queries(queries_path)
model_type = set_model_type(is_complex, model_type)

if "multimer" in model_type and max_msa is not None:
  use_cluster_profile = False
else:
  use_cluster_profile = True

download_alphafold_params(model_type, Path("."))  # 모델 파라미터 다운로드
# 모든 설정을 적용하여 구조 예측 실행
results = run(
    queries=queries,
    result_dir=result_dir,
    use_templates=use_templates,
    custom_template_path=custom_template_path,
    num_relax=num_relax,
    msa_mode=msa_mode,
    model_type=model_type,
    num_models=5,
    num_recycles=num_recycles,
    relax_max_iterations=relax_max_iterations,
    recycle_early_stop_tolerance=recycle_early_stop_tolerance,
    num_seeds=num_seeds,
    use_dropout=use_dropout,
    model_order=[1,2,3,4,5],
    is_complex=is_complex,
    data_dir=Path("."),
    keep_existing_results=False,
    rank_by="auto",
    pair_mode=pair_mode,
    pairing_strategy=pairing_strategy,
    stop_at_score=float(100),
    prediction_callback=prediction_callback,
    dpi=dpi,
    zip_results=False,
    save_all=save_all,
    max_msa=max_msa,
    use_cluster_profile=use_cluster_profile,
    input_features_callback=input_features_callback,
    save_recycles=save_recycles,
    user_agent="colabfold/google-colab-main",
    calc_extra_ptm=calc_extra_ptm,
)
results_zip = f"{jobname}.result.zip"
os.system(f"zip -r {results_zip} {jobname}")  # 결과를 zip으로 압축

In [ ]:
#@title 3D 구조 표시 {run: "auto"}
import py3Dmol
import glob
import matplotlib.pyplot as plt
from colabfold.colabfold import plot_plddt_legend
from colabfold.colabfold import pymol_color_list, alphabet_list
rank_num = 1 #@param ["1", "2", "3", "4", "5"] {type:"raw"}
color = "lDDT" #@param ["chain", "lDDT", "rainbow"]
show_sidechains = False #@param {type:"boolean"}
show_mainchains = False #@param {type:"boolean"}

tag = results["rank"][0][rank_num - 1]  # 선택한 랭크의 태그
jobname_prefix = ".custom" if msa_mode == "custom" else ""
pdb_filename = f"{jobname}/{jobname}{jobname_prefix}_unrelaxed_{tag}.pdb"
pdb_file = glob.glob(pdb_filename)

def show_pdb(rank_num=1, show_sidechains=False, show_mainchains=False, color="lDDT"):
  model_name = f"rank_{rank_num}"
  view = py3Dmol.view(js='https://3dmol.org/build/3Dmol.js',)
  view.addModel(open(pdb_file[0],'r').read(),'pdb')

  # 색상 스키마 적용
  if color == "lDDT":
    view.setStyle({'cartoon': {'colorscheme': {'prop':'b','gradient': 'roygb','min':50,'max':90}}})
  elif color == "rainbow":
    view.setStyle({'cartoon': {'color':'spectrum'}})
  elif color == "chain":
    chains = len(queries[0][1]) + 1 if is_complex else 1
    for n,chain,color in zip(range(chains),alphabet_list,pymol_color_list):
       view.setStyle({'chain':chain},{'cartoon': {'color':color}})

  # 곁사슬 표시
  if show_sidechains:
    BB = ['C','O','N']
    view.addStyle({'and':[{'resn':["GLY","PRO"],'invert':True},{'atom':BB,'invert':True}]},
                        {'stick':{'colorscheme':f"WhiteCarbon",'radius':0.3}})
    view.addStyle({'and':[{'resn':"GLY"},{'atom':'CA'}]},
                        {'sphere':{'colorscheme':f"WhiteCarbon",'radius':0.3}})
    view.addStyle({'and':[{'resn':"PRO"},{'atom':['C','O'],'invert':True}]},
                        {'stick':{'colorscheme':f"WhiteCarbon",'radius':0.3}})
  # 주사슬 표시
  if show_mainchains:
    BB = ['C','O','N','CA']
    view.addStyle({'atom':BB},{'stick':{'colorscheme':f"WhiteCarbon",'radius':0.3}})

  view.zoomTo()
  return view

show_pdb(rank_num, show_sidechains, show_mainchains, color).show()
if color == "lDDT":
  plot_plddt_legend().show()  # lDDT 색상 범례 표시

In [ ]:
#@title 플롯 {run: "auto"}
from IPython.display import display, HTML
import base64
from html import escape

# 참고: https://stackoverflow.com/a/53688522
def image_to_data_url(filename):
  ext = filename.split('.')[-1]
  prefix = f'data:image/{ext};base64,'
  with open(filename, 'rb') as f:
    img = f.read()
  return prefix + base64.b64encode(img).decode('utf-8')

# 생성된 PAE / 커버리지 / pLDDT 플롯 이미지를 불러옴
pae = ""
pae_file = os.path.join(jobname,f"{jobname}{jobname_prefix}_pae.png")
if os.path.isfile(pae_file):
    pae = image_to_data_url(pae_file)
cov = image_to_data_url(os.path.join(jobname,f"{jobname}{jobname_prefix}_coverage.png"))
plddt = image_to_data_url(os.path.join(jobname,f"{jobname}{jobname_prefix}_plddt.png"))
display(HTML(f"""
<style>
  img {{
    float:left;
  }}
  .full {{
    max-width:100%;
  }}
  .half {{
    max-width:50%;
  }}
  @media (max-width:640px) {{
    .half {{
      max-width:100%;
    }}
  }}
</style>
<div style="max-width:90%; padding:2em;">
  <h1>Plots for {escape(jobname)}</h1>
  { '<!--' if pae == '' else '' }<img src="{pae}" class="full" />{ '-->' if pae == '' else '' }
  <img src="{cov}" class="half" />
  <img src="{plddt}" class="half" />
</div>
"""))

In [ ]:
#@title 결과 패키징 및 다운로드
#@markdown 결과 아카이브 다운로드에 문제가 있으면, 광고 차단기를 끄고 이 셀을 다시 실행해 보세요. 그래도 안 되면 왼쪽의 폴더 아이콘을 눌러 `jobname.result.zip` 파일을 찾아 우클릭 → "다운로드"를 선택하세요.

if msa_mode == "custom":
  print("맞춤 MSA 생성 방법을 인용(cite)하는 것을 잊지 마세요.")

files.download(f"{jobname}.result.zip")

if save_to_google_drive == True and drive:
  uploaded = drive.CreateFile({'title': f"{jobname}.result.zip"})
  uploaded.SetContentFile(f"{jobname}.result.zip")
  uploaded.Upload()
  print(f"{jobname}.result.zip 을 Google Drive에 업로드했습니다. ID: {uploaded.get('id')}")

# 사용법 (Instructions) <a name="Instructions"></a>
자세한 사용법과 팁은 최근 발표된 [Nature Protocols](https://www.nature.com/articles/s41596-024-01060-5) 논문을 참고하세요.

**빠른 시작**
1. 입력 칸에 단백질 서열을 붙여넣습니다.
2. `런타임` -> `모두 실행`을 누릅니다.
3. 파이프라인은 5단계로 구성됩니다. 현재 실행 중인 단계는 옆에 정지 표시가 있는 원으로 나타납니다.

**결과 zip 파일 내용**
1. 평균 pLDDT(복합체는 pTMscore) 순으로 정렬된 PDB 구조 (`use_amber` 사용 시 relaxed 포함).
2. 모델 품질 플롯.
3. MSA 커버리지 플롯.
4. 파라미터 로그 파일.
5. A3M 형식의 입력 MSA.
6. [AlphaFold-DB 형식](https://alphafold.ebi.ac.uk/faq#faq-7)의 `predicted_aligned_error_v1.json`과, 각 모델의 PAE 배열·평균 pLDDT·pTMscore를 담은 `scores.json`.
7. 사용한 모든 도구·데이터베이스의 인용을 담은 BibTeX 파일.

작업이 끝나면 `jobname.result.zip` 다운로드 창이 뜹니다. `save_to_google_drive`를 선택했다면 Google Drive에도 업로드됩니다.

**복합체용 MSA 생성**
복합체 예측에는 짝짓지 않은(unpaired) MSA와 짝지은(paired) MSA를 모두 사용합니다. unpaired MSA는 단백질 구조 예측과 동일하게 UniRef100과 환경 서열을 각각 3회 반복 검색해 생성합니다. paired MSA는 UniRef100을 검색하고 같은 NCBI 분류학적 식별자(종/아종)를 공유하는 최적 히트를 짝지어 생성합니다.

**맞춤 MSA 사용**
맞춤 MSA(A3M 형식)로 예측하려면: (1) `msa_mode`를 "custom"으로 변경, (2) "MSA 옵션" 박스 끝에 업로드 창이 뜨면 A3M을 업로드합니다. A3M의 첫 fasta 항목은 gap 없는 쿼리 서열이어야 합니다.

**PDB100** <a name="pdb100"></a>
2023/06/08부터 PDB70 대신 100% 클러스터링된 PDB100을 사용합니다. PDB100은 각 대표 구조를 [Foldseek](https://github.com/steineggerlab/foldseek)으로 [AlphaFold Database](https://alphafold.ebi.ac.uk)에 검색해 구축합니다. (호환성을 위해 파일/응답 이름은 여전히 "PDB70"으로 표기될 수 있습니다.)

**맞춤 템플릿 사용** <a name="custom_templates"></a>
맞춤 템플릿(PDB/mmCIF)으로 예측하려면: (1) `template_mode`를 "custom"으로 변경, (2) "Input Protein" 박스 끝에 업로드 창이 뜨면 템플릿을 업로드합니다(여러 개 가능).
* 템플릿은 소문자 4글자 PDB 명명을 따라야 합니다.
* mmCIF 템플릿은 `_entity_poly_seq`를 포함해야 합니다. PDB 형식은 자동으로 mmCIF로 변환됩니다.

**문제 해결(Troubleshooting)**
* `런타임` -> `런타임 유형 변경`에서 GPU가 설정돼 있는지 확인하세요.
* `런타임` -> `런타임 초기화(Factory reset runtime)`로 세션을 재시작해 보세요.
* 입력 서열을 확인하세요.

**알려진 이슈**
* Colab은 메모리가 다른 GPU를 할당합니다. 긴 서열은 메모리가 부족할 수 있습니다.
* 브라우저가 결과 다운로드 팝업을 막을 수 있습니다. `save_to_google_drive`를 쓰거나 왼쪽 폴더에서 직접 다운로드하세요.

**한계(Limitations)**
* 계산 자원: MMseqs2 API는 하루 약 2~5만 요청을 처리합니다.
* MSA: MMseqs2는 정밀하지만 BFD/MGnify 대상 HHblits/HMMer보다 히트가 적을 수 있습니다.
* 가능하면 전체 [AlphaFold2 파이프라인](https://github.com/deepmind/alphafold)도 함께 쓰는 것을 권장합니다.

**플롯 설명**
*   **위치별 서열 수(Number of sequences per position)** — 위치마다 최소 30개(이상적으로 100개)의 서열이 있는 것이 좋습니다.
*   **위치별 예측 lDDT** — 각 위치의 모델 신뢰도(100점 만점). 높을수록 좋습니다.
*   **예측 정렬 오차(PAE)** — 동종올리고머에서 계면 신뢰도 평가에 유용. 낮을수록 좋습니다.

**버그**
- 버그는 https://github.com/sokrypton/ColabFold/issues 에 보고해 주세요.

**라이선스**
ColabFold 소스 코드는 [MIT](https://raw.githubusercontent.com/sokrypton/ColabFold/main/LICENSE) 라이선스입니다. 이 노트북은 [Apache 2.0](https://raw.githubusercontent.com/deepmind/alphafold/main/LICENSE)과 [CC BY 4.0](https://creativecommons.org/licenses/by-sa/4.0/) 라이선스의 AlphaFold2 코드·파라미터를 사용합니다.

**감사의 글(Acknowledgments)**
- 훌륭한 모델을 개발하고 오픈소스로 공개한 AlphaFold 팀에 감사드립니다.
- MMseqs2 MSA 서버 자원을 제공한 [KOBIC](https://kobic.re.kr)와 [Söding Lab](https://www.mpinat.mpg.de/soeding).
- 멋진 [py3Dmol](https://3dmol.csb.pitt.edu/) 플러그인을 만든 [David Koes](https://github.com/dkoes).
- ColabFold 로고를 만든 Do-Yoon Kim.
- Sergey Ovchinnikov ([@sokrypton](https://twitter.com/sokrypton)), Milot Mirdita ([@milot_mirdita](https://twitter.com/milot_mirdita)), Martin Steinegger ([@thesteinegger](https://twitter.com/thesteinegger))가 만든 colab입니다.